# LeetCode Hot 100 - Day 22

## 今日主题：二维 DP 与字符串 DP

今天进入二维动态规划，也就是**表格填数**：

1. 最长回文子串：`dp[i][j]` 表示子串 s[i..j] 是不是回文。
2. 不同路径：网格上从左上走到右下，`dp[i][j]` 表示到达方式数。
3. 最小路径和：同样的网格，换成求最小值。
4. 最长公共子序列：两个字符串的经典二维 DP，加练题。

二维 DP 的关键是**填表顺序**：填 `dp[i][j]` 时，它依赖的格子必须已经算好了。


## 今天怎么学

1. 每道题都先把 dp 表格画出来（两三行就够），标出依赖关系，再写循环。
2. 不同路径和最小路径和是同一个格子模型，可以对比着写。
3. 最长公共子序列是“两个字符串对比”的模板，面试高频。

最低目标：独立写出不同路径和最小路径和。


## 今日题单

1. 最长回文子串（LeetCode 5，中等，必做）
2. 不同路径（LeetCode 62，中等，必做）
3. 最小路径和（LeetCode 64，中等，必做）
4. 最长公共子序列（LeetCode 1143，中等，加练）


## 昨日复习

先用 `day21_practice.ipynb` 重写：

1. 零钱兑换。
2. 最长递增子序列。

口述：为什么零钱兑换要用 `amount + 1` 当初始值？


## 二维 DP 的三条规矩

**一、`dp` 是一个二维表格。** 通常写作 `dp[i][j]`，要先用清楚的语言说出它表示什么。

**二、填每一格之前，它依赖的格子必须已经填好。** 常见两种情况：

- 依赖“左上角”，就从上到下、从左到右填（不同路径、最小路径和、最长公共子序列）；
- 依赖“更短的区间”，就按区间长度从小到大填（最长回文子串）。

**三、边界要单独处理。** 通常是第一行和第一列。

下面先手工填一次最小的表格，看清依赖关系。


In [ ]:
# 手工填一次 3 行 3 列的不同路径表
rows = 3
cols = 3
dp = []
for i in range(rows):
    dp.append([1] * cols)

for i in range(1, rows):
    for j in range(1, cols):
        dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
        print("dp[" + str(i) + "][" + str(j) + "] =", dp[i - 1][j], "+", dp[i][j - 1], "=", dp[i][j])

print("表格：")
for line in dp:
    print("   ", line)


## 题目 1 做题前先补：区间型 DP 的填表顺序

`dp[i][j]` 表示“子串 `s[i]` 到 `s[j]` 是不是回文”，是个布尔值。

转移：如果 `s[i] == s[j]`，那么它是不是回文，取决于里面那一层 `dp[i+1][j-1]`。

注意依赖关系：**`dp[i][j]` 依赖的是“更短”的区间**（去掉两头的 `i+1, j-1`）。所以填表顺序不能按行来，要**按区间长度从小到大**：

- 长度 1：单个字符，全是回文；
- 长度 2：两个字符相同就是回文；
- 长度 3 及以上：`s[i] == s[j]` 且 `dp[i+1][j-1]` 为真。

按长度填，就能保证用到的“内部区间”都已经算好了。


In [ ]:
s = "abba"
n = len(s)
dp = []
for i in range(n):
    dp.append([False] * n)

for size in range(1, n + 1):
    for i in range(n - size + 1):
        j = i + size - 1
        if size == 1:
            dp[i][j] = True
        elif size == 2:
            if s[i] == s[j]:
                dp[i][j] = True
        else:
            if s[i] == s[j] and dp[i + 1][j - 1]:
                dp[i][j] = True
        if dp[i][j]:
            print("区间", i, "到", j, "（", s[i:j + 1], "）是回文")


# 题目 1：最长回文子串

LeetCode 5. Longest Palindromic Substring

## 题目描述（改写版）

给你一个字符串 `s`，请找出其中最长的**回文子串**，返回这个子串本身。

回文串是正着读和反着读一样的字符串。子串必须是连续的。

## 输入

- `s`：长度 1 到 1000，只含数字和英文字母。

## 输出

返回最长的回文子串。如果有多个答案，返回任意一个。

## 示例

示例 1：`s = "babad"`，返回 `"bab"` 或 `"aba"`。

示例 2：`s = "cbbd"`，返回 `"bb"`。

示例 3：`s = "a"`，返回 `"a"`。

## 易漏细节

- 是子串不是子序列，必须连续。
- 记录答案时存“起点 + 长度”两个变量，不要每次都存字符串。
- 长度 1 和长度 2 要单独初始化。
- 一个字符也算回文，所以答案至少是 1 个字符。


## 解法名称

**区间动态规划（Interval DP）**；另一条路是**中心扩展法（Expand Around Center）**。

## 暴力思路

枚举所有子串（O(n²) 个），每个判断是否回文（O(n)），总时间 O(n³)。n = 1000 时太慢。

## 优化思路

区间 DP：

- `dp[i][j]`：`s[i..j]` 是否回文；
- 长度 1：全部为真；
- 长度 2：`s[i] == s[j]` 为真；
- 长度 ≥ 3：`s[i] == s[j] and dp[i+1][j-1]`；
- 每次为真时，用区间长度更新答案。

时间 O(n²)，空间 O(n²)。

**中心扩展法**（面试加分）：枚举每个中心（n 个奇中心 + n-1 个偶中心），向两边扩展，时间也是 O(n²)，但空间 O(1)。两个都值得会。


## 你来写：最长回文子串

要求：

- 用区间 DP 写，按长度从小到大填表。
- 用 `start` 和 `length` 记录答案。
- 写完用 `"babad"`、`"cbbd"`、`"a"` 各跑一遍。

先在心里回答：为什么不能不按长度、直接按行列填表？


In [ ]:
# 题目：最长回文子串
# 解法：区间动态规划（Interval DP）
# 输入：字符串 s，长度 1 到 1000。
# 目标：找出最长的回文子串（连续）。
# 输出：返回这个子串。
# 注意：dp[i][j] 表示 s[i..j] 是否回文；按区间长度从小到大填；用 start 和 length 记录答案。


def longest_palindrome(s):
    # 在这里写你的代码
    pass


print(longest_palindrome("babad"))


In [ ]:
result = longest_palindrome("babad")
if result is None:
    print("longest_palindrome 还没有返回结果，先把上面的函数写完再运行这一格。")
else:
    print(result)                          # 期望 bab 或 aba
    print(longest_palindrome("cbbd"))      # 期望 bb
    print(longest_palindrome("a"))         # 期望 a
    print(longest_palindrome("aaaa"))      # 期望 aaaa
    print(longest_palindrome("abc"))       # 期望 a（或 b、c，长度都是 1）


## 参考答案：最长回文子串

```python
def longest_palindrome_answer(s):
    n = len(s)
    dp = []
    for i in range(n):
        dp.append([False] * n)

    start = 0
    length = 1

    for size in range(1, n + 1):
        for i in range(n - size + 1):
            j = i + size - 1
            if size == 1:
                dp[i][j] = True
            elif size == 2:
                if s[i] == s[j]:
                    dp[i][j] = True
            else:
                if s[i] == s[j] and dp[i + 1][j - 1]:
                    dp[i][j] = True

            if dp[i][j] and size > length:
                start = i
                length = size

    return s[start:start + length]
```

面试表达：

我用区间动态规划。`dp[i][j]` 表示从 i 到 j 的子串是不是回文。单个字符一定是回文；两个字符只要相等就是回文；更长的话，需要两端字符相等，并且去掉两端后的内部子串也是回文，也就是 `dp[i+1][j-1]` 为真。因为用到了更短的区间，所以填表要按区间长度从小到大进行。每次发现回文就用长度更新答案的起点和长度，最后切片返回。时间 O(n 的平方)，空间也是 O(n 的平方)。

另外也可以用中心扩展法：枚举每个字符和每两个字符之间作为中心，向两边扩展，时间一样是 O(n 的平方)，但空间只要 O(1)。


In [ ]:


def longest_palindrome_answer(s):
    n = len(s)
    dp = []
    for i in range(n):
        dp.append([False] * n)

    start = 0
    length = 1

    for size in range(1, n + 1):
        for i in range(n - size + 1):
            j = i + size - 1
            if size == 1:
                dp[i][j] = True
            elif size == 2:
                if s[i] == s[j]:
                    dp[i][j] = True
            else:
                if s[i] == s[j] and dp[i + 1][j - 1]:
                    dp[i][j] = True

            if dp[i][j] and size > length:
                start = i
                length = size

    return s[start:start + length]


print(longest_palindrome_answer("babad"))    # bab 或 aba
print(longest_palindrome_answer("cbbd"))     # bb
print(longest_palindrome_answer("a"))        # a
print(longest_palindrome_answer("aaaa"))     # aaaa


## 题目 2 做题前先补：到达某一格只有两条路

网格上从左上走到右下，只能向右或向下。

**到达 `(i, j)` 这一格，只可能来自两个地方：**

- 从上面 `(i-1, j)` 走下来；
- 从左边 `(i, j-1)` 走过来。

所以 `dp[i][j] = dp[i-1][j] + dp[i][j-1]`，这就是“路径数相加”。

边界：第一行只能一直往右走（只有 1 种走法），第一列只能一直往下走（也只有 1 种）。所以第一行和第一列全部是 1。

填表顺序：从上到下、从左到右（因为依赖上面和左边）。


In [ ]:
m = 3
n = 4
dp = []
for i in range(m):
    dp.append([1] * n)

for i in range(1, m):
    for j in range(1, n):
        dp[i][j] = dp[i - 1][j] + dp[i][j - 1]

print("路径数表格：")
for line in dp:
    print("   ", line)
print("从左上到右下的路径数：", dp[m - 1][n - 1])


# 题目 2：不同路径

LeetCode 62. Unique Paths

## 题目描述（改写版）

一个机器人站在 `m × n` 网格的左上角，每次只能向右或向下移动一步。请问到达右下角一共有多少条不同的路径。

## 输入

- `m`、`n`：1 到 100。

## 输出

返回路径总数。

## 示例

示例 1：`m = 3`，`n = 7`，返回 28。

示例 2：`m = 3`，`n = 2`，返回 3。

示例 3：`m = 1`，`n = 1`，返回 1。

## 易漏细节

- 第一行和第一列都只有 1 种走法。
- 只有一行或一列时答案就是 1。
- 路径数增长很快，但题目数据范围下不会溢出。


## 解法名称

**二维动态规划（2D DP）**，也叫“网格递推”。

## 暴力思路

递归枚举每一步往右还是往下，分支数量是组合数级别，重复子问题极多。

## 优化思路

- `dp[i][j]`：从起点走到 `(i, j)` 的路径数；
- 第一行、第一列全部是 1；
- 其他格子：`dp[i][j] = dp[i-1][j] + dp[i][j-1]`；
- 答案是 `dp[m-1][n-1]`。

时间 O(m × n)，空间 O(m × n)。也可以用一维数组滚动优化到 O(n)。


## 你来写：不同路径

要求：

- 用二维 DP 写，第一行第一列初始化成 1。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么第一行和第一列都是 1？


In [ ]:
# 题目：不同路径
# 解法：二维动态规划（2D DP）
# 输入：网格行数 m 和列数 n（都是 1 到 100）。
# 目标：机器人从左上角出发，每次只能向右或向下，求到达右下角的路径数。
# 输出：返回路径总数（整数）。
# 注意：dp[i][j] = dp[i-1][j] + dp[i][j-1]；第一行和第一列都是 1；从上到下、从左到右填表。


def unique_paths(m, n):
    # 在这里写你的代码
    pass


print(unique_paths(3, 7))


In [ ]:
print(unique_paths(3, 7))     # 期望 28
print(unique_paths(3, 2))     # 期望 3
print(unique_paths(1, 1))     # 期望 1
print(unique_paths(1, 10))    # 期望 1
print(unique_paths(7, 3))     # 期望 28


## 参考答案：不同路径

```python
def unique_paths_answer(m, n):
    dp = []
    for i in range(m):
        dp.append([1] * n)

    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = dp[i - 1][j] + dp[i][j - 1]

    return dp[m - 1][n - 1]
```

面试表达：

我用二维动态规划，`dp[i][j]` 表示从左上角走到这一格的路径数。因为每次只能向右或向下，所以到达某一格只能从上面或左边来，路径数就是两个方向的路径数之和。第一行只能一直往右走、第一列只能一直往下走，所以它们都是 1。然后从上到下、从左到右把表填完，右下角就是答案。时间 O(m 乘 n)，空间 O(m 乘 n)；如果只想省空间，也可以只用一维数组滚动更新。


In [ ]:


def unique_paths_answer(m, n):
    dp = []
    for i in range(m):
        dp.append([1] * n)

    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = dp[i - 1][j] + dp[i][j - 1]

    return dp[m - 1][n - 1]


print(unique_paths_answer(3, 7))     # 28
print(unique_paths_answer(3, 2))     # 3
print(unique_paths_answer(1, 1))     # 1
print(unique_paths_answer(7, 3))     # 28


## 题目 3 做题前先补：把“求和”换成“求最小”

和上一题的模型几乎一样：从左上走到右下，只能向右或向下，只是这次每格有代价，要求总代价最小。

`dp[i][j]` 的定义变成：**走到 `(i, j)` 的最小路径和。**

转移也是两个来源，但这次取**较小**的那个：

```text
dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + grid[i][j]
```

边界也要加上代价：

- 起点 `dp[0][0] = grid[0][0]`；
- 第一行：只能从左边来，`dp[0][j] = dp[0][j-1] + grid[0][j]`；
- 第一列：只能从上边来，`dp[i][0] = dp[i-1][0] + grid[i][0]`。

**对比记忆：不同路径是“相加”，最小路径和是“取小再加”。**


In [ ]:
grid = [
    [1, 3, 1],
    [1, 5, 1],
    [4, 2, 1],
]
rows = len(grid)
cols = len(grid[0])
dp = []
for i in range(rows):
    dp.append([0] * cols)

dp[0][0] = grid[0][0]
for j in range(1, cols):
    dp[0][j] = dp[0][j - 1] + grid[0][j]
for i in range(1, rows):
    dp[i][0] = dp[i - 1][0] + grid[i][0]

for i in range(1, rows):
    for j in range(1, cols):
        if dp[i - 1][j] < dp[i][j - 1]:
            dp[i][j] = dp[i - 1][j] + grid[i][j]
        else:
            dp[i][j] = dp[i][j - 1] + grid[i][j]

print("最小路径和表格：")
for line in dp:
    print("   ", line)
print("答案：", dp[rows - 1][cols - 1])


# 题目 3：最小路径和

LeetCode 64. Minimum Path Sum

## 题目描述（改写版）

给你一个非负整数网格 `grid`，从左上角出发，每次只能向右或向下，走到右下角。请返回一条路径上数字之和的**最小值**。

## 输入

- `grid`：1 到 200 行，1 到 200 列，元素 0 到 200。

## 输出

返回最小路径和。

## 示例

示例 1：

```text
1 3 1
1 5 1
4 2 1
```

最优路径是 `1 -> 3 -> 1 -> 1 -> 1`，和是 7。

示例 2：只有一格 `[[5]]`，返回 5。

## 易漏细节

- 起点也要计入代价。
- 第一行和第一列只能沿一个方向来，要累加。
- 每个格子只依赖上面和左边，所以从上到下、从左到右填。


## 解法名称

**二维动态规划（2D DP）**。

## 暴力思路

递归枚举所有向右/向下的组合，分支数量是组合数级别。

## 优化思路

- `dp[i][j]`：走到 `(i, j)` 的最小路径和；
- 起点 `dp[0][0] = grid[0][0]`；
- 第一行累加，第一列累加；
- 其他格子：`dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + grid[i][j]`。

时间 O(m × n)，空间 O(m × n)。


## 你来写：最小路径和

要求：

- 用二维 DP 写，注意起点和两条边界的初始化。
- 中间用 `min` 的思路（可以写成 `if` 判断，也可以直接比较）。
- 写完用示例的两组数据各跑一遍。

先在心里回答：为什么这道题的边界不能像“不同路径”那样全部初始化成 1？


In [ ]:
# 题目：最小路径和
# 解法：二维动态规划（2D DP）
# 输入：非负整数网格 grid，行列数 1 到 200。
# 目标：从左上角走到右下角，只能向右或向下，求路径上数字和的最小值。
# 输出：返回最小路径和（整数）。
# 注意：起点 dp[0][0] = grid[0][0]；第一行和第一列要累加；其他格取上面和左边较小的再加自己。


def min_path_sum(grid):
    # 在这里写你的代码
    pass


grid = [
    [1, 3, 1],
    [1, 5, 1],
    [4, 2, 1],
]
print(min_path_sum(grid))


In [ ]:
grid1 = [[1, 3, 1], [1, 5, 1], [4, 2, 1]]
print(min_path_sum(grid1))          # 期望 7

print(min_path_sum([[5]]))          # 期望 5
print(min_path_sum([[1, 2, 3]]))    # 期望 6
print(min_path_sum([[1], [2], [3]]))    # 期望 6
print(min_path_sum([[0, 0], [0, 0]]))   # 期望 0


## 参考答案：最小路径和

```python
def min_path_sum_answer(grid):
    rows = len(grid)
    cols = len(grid[0])
    dp = []
    for i in range(rows):
        dp.append([0] * cols)

    dp[0][0] = grid[0][0]
    for j in range(1, cols):
        dp[0][j] = dp[0][j - 1] + grid[0][j]
    for i in range(1, rows):
        dp[i][0] = dp[i - 1][0] + grid[i][0]

    for i in range(1, rows):
        for j in range(1, cols):
            if dp[i - 1][j] < dp[i][j - 1]:
                dp[i][j] = dp[i - 1][j] + grid[i][j]
            else:
                dp[i][j] = dp[i][j - 1] + grid[i][j]

    return dp[rows - 1][cols - 1]
```

面试表达：

我用二维动态规划，`dp[i][j]` 表示走到这一格的最小路径和。因为只能向右或向下，到达某格只能从上面或左边来，所以取两者中较小的路径和，再加上当前格子的值。起点是它自己的值；第一行只能从左边一路累加，第一列只能从上边一路累加。填表顺序是从上到下、从左到右，最后返回右下角。时间 O(m 乘 n)，空间 O(m 乘 n)。


In [ ]:


def min_path_sum_answer(grid):
    rows = len(grid)
    cols = len(grid[0])
    dp = []
    for i in range(rows):
        dp.append([0] * cols)

    dp[0][0] = grid[0][0]
    for j in range(1, cols):
        dp[0][j] = dp[0][j - 1] + grid[0][j]
    for i in range(1, rows):
        dp[i][0] = dp[i - 1][0] + grid[i][0]

    for i in range(1, rows):
        for j in range(1, cols):
            if dp[i - 1][j] < dp[i][j - 1]:
                dp[i][j] = dp[i - 1][j] + grid[i][j]
            else:
                dp[i][j] = dp[i][j - 1] + grid[i][j]

    return dp[rows - 1][cols - 1]


print(min_path_sum_answer([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))    # 7
print(min_path_sum_answer([[5]]))          # 5
print(min_path_sum_answer([[1, 2, 3]]))    # 6
print(min_path_sum_answer([[1], [2], [3]]))    # 6


## 题目 4 做题前先补：两个字符串的对比表

最长公共子序列（LCS）是二维 DP 的经典模型：给两个字符串，求它们**共同**的最长子序列长度（子序列不要求连续）。

状态定义：`dp[i][j]` = `text1` 的前 i 个字符和 `text2` 的前 j 个字符的 LCS 长度。

转移分两种情况：

- 如果 `text1[i-1] == text2[j-1]`：这两个字符能配上，`dp[i][j] = dp[i-1][j-1] + 1`；
- 否则：两个字符配不上，只能丢掉其中一个，`dp[i][j] = max(dp[i-1][j], dp[i][j-1])`。

**注意下标**：`dp` 的第 i 行对应 `text1[i-1]`，所以比较时用 `text1[i-1]` 和 `text2[j-1]`，这是最容易写错的地方。

答案在 `dp[m][n]`。可以说：**LCS 是“两个字符串对比表”，后面编辑距离也是这张表的变体。**


In [ ]:
text1 = "abcde"
text2 = "ace"
m = len(text1)
n = len(text2)
dp = []
for i in range(m + 1):
    dp.append([0] * (n + 1))

for i in range(1, m + 1):
    for j in range(1, n + 1):
        if text1[i - 1] == text2[j - 1]:
            dp[i][j] = dp[i - 1][j - 1] + 1
        else:
            if dp[i - 1][j] > dp[i][j - 1]:
                dp[i][j] = dp[i - 1][j]
            else:
                dp[i][j] = dp[i][j - 1]

print("LCS 长度表格：")
for line in dp:
    print("   ", line)
print("答案：", dp[m][n])


# 题目 4：最长公共子序列

LeetCode 1143. Longest Common Subsequence

## 题目描述（改写版）

给你两个字符串 `text1` 和 `text2`，请返回它们**最长公共子序列**的长度。如果不存在公共子序列，返回 0。

子序列是指删掉若干字符后剩下的字符，**不要求连续**，但要保持原来的先后顺序。

## 输入

- `text1`、`text2`：长度 1 到 1000，只含小写英文字母。

## 输出

返回最长公共子序列的长度。

## 示例

示例 1：`text1 = "abcde"`，`text2 = "ace"`，返回 3（"ace"）。

示例 2：`text1 = "abc"`，`text2 = "abc"`，返回 3。

示例 3：`text1 = "abc"`，`text2 = "def"`，返回 0。

## 易漏细节

- 子序列不要求连续。
- `dp` 要比字符串长度多开一位，第 0 行第 0 列表示“空串”。
- 比较的是 `text1[i-1]` 和 `text2[j-1]`。


## 解法名称

**二维动态规划（2D DP）**。

## 暴力思路

枚举 `text1` 的所有子序列，检查是否也是 `text2` 的子序列，指数级。

## 优化思路

- `dp[i][j]`：前 i 个和前 j 个字符的 LCS 长度；
- 第 0 行、第 0 列全是 0（空串没有公共子序列）；
- 字符相同：`dp[i][j] = dp[i-1][j-1] + 1`；
- 字符不同：`dp[i][j] = max(dp[i-1][j], dp[i][j-1])`；
- 答案 `dp[m][n]`。

时间 O(m × n)，空间 O(m × n)。


## 你来写：最长公共子序列

要求：

- 用二维 DP 写，注意下标从 1 开始、比较时用 `i-1` 和 `j-1`。
- 写完用示例的三组数据各跑一遍。

先在心里回答：字符不相同时，为什么答案取“丢掉某一个字符”的最大值？


In [ ]:
# 题目：最长公共子序列
# 解法：二维动态规划（2D DP）
# 输入：两个只含小写字母的字符串 text1 和 text2，长度 1 到 1000。
# 目标：求最长公共子序列的长度（不要求连续）。
# 输出：返回长度（整数）。
# 注意：dp 行列都比字符串长 1，表示空串；比较 text1[i-1] 和 text2[j-1]；不同时取上面和左边的较大值。


def longest_common_subsequence(text1, text2):
    # 在这里写你的代码
    pass


print(longest_common_subsequence("abcde", "ace"))


In [ ]:
print(longest_common_subsequence("abcde", "ace"))    # 期望 3
print(longest_common_subsequence("abc", "abc"))      # 期望 3
print(longest_common_subsequence("abc", "def"))      # 期望 0
print(longest_common_subsequence("a", "a"))          # 期望 1
print(longest_common_subsequence("bl", "yby"))       # 期望 1
print(longest_common_subsequence("abcba", "abcbcba"))    # 期望 5


## 参考答案：最长公共子序列

```python
def longest_common_subsequence_answer(text1, text2):
    m = len(text1)
    n = len(text2)
    dp = []
    for i in range(m + 1):
        dp.append([0] * (n + 1))

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                if dp[i - 1][j] > dp[i][j - 1]:
                    dp[i][j] = dp[i - 1][j]
                else:
                    dp[i][j] = dp[i][j - 1]

    return dp[m][n]
```

面试表达：

我定义 `dp[i][j]` 为 `text1` 前 i 个字符和 `text2` 前 j 个字符的最长公共子序列长度。表格比字符串多一行一列，表示空串的情况。填表时，如果两个当前字符相同，说明它们可以配成一对，答案就是左上角加一；如果不同，说明这两个字符至少有一个用不上，那就取“丢掉 text1 当前字符”和“丢掉 text2 当前字符”两种情况里更大的那个。最后返回右下角。时间 O(m 乘 n)，空间 O(m 乘 n)。这题也是编辑距离的基础，两张表结构一样。


In [ ]:


def longest_common_subsequence_answer(text1, text2):
    m = len(text1)
    n = len(text2)
    dp = []
    for i in range(m + 1):
        dp.append([0] * (n + 1))

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                if dp[i - 1][j] > dp[i][j - 1]:
                    dp[i][j] = dp[i - 1][j]
                else:
                    dp[i][j] = dp[i][j - 1]

    return dp[m][n]


print(longest_common_subsequence_answer("abcde", "ace"))    # 3
print(longest_common_subsequence_answer("abc", "abc"))      # 3
print(longest_common_subsequence_answer("abc", "def"))      # 0
print(longest_common_subsequence_answer("bl", "yby"))       # 1
print(longest_common_subsequence_answer("abcba", "abcbcba"))    # 5


# 今日小结

今天四个二维 DP 模型：

1. **最长回文子串**：按区间长度从小到大填，`dp[i][j]` 靠 `dp[i+1][j-1]`。
2. **不同路径**：`dp[i][j] = 上面 + 左边`，边界都是 1。
3. **最小路径和**：`dp[i][j] = min(上面, 左边) + 当前格`，注意起点和边界要累加。
4. **最长公共子序列**：表格多一行一列表示空串；相同取左上角加一，不同取上面和左边的较大值。

记住填表顺序的口诀：**依赖上面和左边就从左上往右下填；依赖更小区间就按长度从小到大填。**


## 今日复盘区

- 最长回文子串为什么必须按长度填表？
- 不同路径的第一行第一列为什么都是 1？
- 最小路径和的起点为什么要单独初始化？
- 最长公共子序列的表格为什么要多开一行一列？
- 今天哪几道题能不看答案写出来？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
